## Retail Sales Prediction

In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Retail Sales Prediction - MLflow
# MAGIC # Linear Regression

# COMMAND ----------

# 1. IMPORT LIBRARIES

import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from mlflow.models import infer_signature

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# COMMAND ----------

# 2. READ GOLD TABLE

d = spark.table(
    "capretailnew.gold.gold_daily_sales_summary"
).select(
    "sales_date",
    "store_id",
    "category",
    "sales_after_discount_before_tax"
).toPandas()

# COMMAND ----------

# 3. DATA PREPARATION

d["sales_date"] = pd.to_datetime(d["sales_date"])

d["store_id"] = d["store_id"].astype(str)

d["category"] = d["category"].astype(str)

d["net_sales"] = pd.to_numeric(
    d["sales_after_discount_before_tax"],
    errors="coerce"
)

d = d.dropna(
    subset=["sales_date", "store_id", "category", "net_sales"]
)

d = d.sort_values(
    ["store_id", "category", "sales_date"]
).reset_index(drop=True)

# COMMAND ----------

# 4. FEATURE ENGINEERING

group_columns = ["store_id", "category"]

grouped = d.groupby(group_columns)["net_sales"]

d["previous_sales"] = grouped.shift(1)

d["previous_2_sales"] = grouped.shift(2)

d["previous_3_sales"] = grouped.shift(3)

d["rolling_mean_3"] = grouped.transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

d["rolling_mean_7"] = grouped.transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).mean()
)

d["days_gap"] = d.groupby(group_columns)["sales_date"].diff().dt.days

d["day_of_week"] = d["sales_date"].dt.dayofweek

d["month"] = d["sales_date"].dt.month

d["day_sin"] = np.sin(2 * np.pi * d["day_of_week"] / 7)

d["day_cos"] = np.cos(2 * np.pi * d["day_of_week"] / 7)

d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12)

d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)

d["is_weekend"] = (d["day_of_week"] >= 5).astype(int)

d = d.dropna(
    subset=["previous_sales", "days_gap", "net_sales"]
).copy()

# COMMAND ----------

# 5. SELECT FEATURES

numeric_features = [
    "previous_sales",
    "previous_2_sales",
    "previous_3_sales",
    "rolling_mean_3",
    "rolling_mean_7",
    "days_gap",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "is_weekend"
]

categorical_features = [
    "store_id",
    "category"
]

features = numeric_features + categorical_features

X = d[features].copy()

y = d["net_sales"].copy()

X[numeric_features] = X[numeric_features].astype("float64")

# COMMAND ----------

# 6. CHRONOLOGICAL TRAIN / TEST SPLIT

unique_dates = sorted(d["sales_date"].unique())

split_index = int(len(unique_dates) * 0.8)

split_date = unique_dates[split_index]

train = d[d["sales_date"] < split_date].copy()

test = d[d["sales_date"] >= split_date].copy()

X_train = train[features].copy()

X_test = test[features].copy()

y_train = train["net_sales"].copy()

y_test = test["net_sales"].copy()

print("Training records:", len(train))

print("Testing records:", len(test))

print("Split date:", split_date)

# COMMAND ----------

# 7. PREPROCESSING

numeric_processor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_processor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_processor, numeric_features),
        ("categorical", categorical_processor, categorical_features)
    ]
)

# COMMAND ----------

# 8. LINEAR REGRESSION MODEL

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

model.fit(X_train, y_train)

print("Model training completed!")

# COMMAND ----------

# 9. PREDICTION

pred = model.predict(X_test)

# COMMAND ----------

# 10. MODEL EVALUATION

mae = mean_absolute_error(y_test, pred)

rmse = np.sqrt(mean_squared_error(y_test, pred))

r2 = r2_score(y_test, pred)

# COMMAND ----------

# 11. BASELINE COMPARISON

mean_predictions = np.full(
    len(y_test),
    y_train.mean()
)

mean_mae = mean_absolute_error(
    y_test,
    mean_predictions
)

mean_rmse = np.sqrt(
    mean_squared_error(y_test, mean_predictions)
)

previous_predictions = X_test["previous_sales"].values

previous_mae = mean_absolute_error(
    y_test,
    previous_predictions
)

previous_rmse = np.sqrt(
    mean_squared_error(y_test, previous_predictions)
)

# COMMAND ----------

# 12. DISPLAY RESULTS

comparison = pd.DataFrame({
    "Model": [
        "Training Mean Baseline",
        "Previous Sales Baseline",
        "Linear Regression"
    ],
    "MAE": [
        mean_mae,
        previous_mae,
        mae
    ],
    "RMSE": [
        mean_rmse,
        previous_rmse,
        rmse
    ]
})

display(comparison)

print("Linear Regression R2:", r2)

# COMMAND ----------

# 13. MODEL SIGNATURE

input_example = X_train.head(5).copy()

output_example = model.predict(input_example)

signature = infer_signature(
    input_example,
    output_example
)

# COMMAND ----------

# 14. MLFLOW TRACKING

mlflow.set_experiment(
    "/Shared/retail_sales_prediction"
)

with mlflow.start_run(
    run_name="Retail_Final_Linear_Regression"
) as run:

    mlflow.log_params({
        "model": "LinearRegression",
        "features": ",".join(features),
        "split": "80/20 chronological",
        "split_date": str(pd.Timestamp(split_date).date()),
        "training_records": len(train),
        "testing_records": len(test)
    })

    mlflow.log_metrics({
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "mean_baseline_MAE": mean_mae,
        "mean_baseline_RMSE": mean_rmse,
        "previous_sales_baseline_MAE": previous_mae,
        "previous_sales_baseline_RMSE": previous_rmse
    })

    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=input_example
    )

    model_uri = f"runs:/{run.info.run_id}/model"

    print("MLflow Run ID:", run.info.run_id)

    print("Model URI:", model_uri)

# COMMAND ----------

# 15. LOAD SAVED MODEL

loaded_model = mlflow.sklearn.load_model(model_uri)

loaded_predictions = loaded_model.predict(X_test)

print("Saved model loaded successfully!")

# COMMAND ----------

# 16. SAMPLE PREDICTIONS

results = pd.DataFrame({
    "Actual_Sales": y_test.values,
    "Predicted_Sales": loaded_predictions
})

display(results.head(10))

# COMMAND ----------

# 17. FINAL STATUS

print("====================================")
print("FINAL MLFLOW RESULTS")
print("====================================")

print("Training records:", len(train))
print("Testing records:", len(test))

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

print("Model:", "Linear Regression")
print("MLflow tracking completed!")
print("Model logging completed!")
print("Model loading completed!")
